<table>
<tr><td><img style="height: 150px;" src="images/geo_hydro1.jpg"></td>
<td bgcolor="#FFFFFF">
    <p style="font-size: xx-large; font-weight: 900; line-height: 100%">AG Dynamics of the Earth</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Juypter notebooks</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Georg Kaufmann</p>
    </td>
</tr>
</table>

# 05: Logistic regression
----
*Georg Kaufmann,
Geophysics Section,
Institute of Geological Sciences,
Freie Universität Berlin,
Germany*

In [ ]:
import numpy as np
import scipy.optimize
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib import cm
from ipywidgets import interact, interactive, fixed
import ipywidgets as widgets

## Preparation: Create color bars

Create different color maps ...

In [ ]:
# color bars
n_colors = 9
mycolors_bold = ['#FF0000', '#0000FF', '#00FF00']
mycolors_light = ['#FFAAAA','#AAAAFF', '#AAFFAA']
cmap_bold = ListedColormap(mycolors_bold)
cmap_light = ListedColormap(mycolors_light)
cmap_set1 = cm.get_cmap('Set1', 9) # Set1,Set2,Set3

## Preparation: Create random points 

Create random points $X_{ij}, i=1,n; j=1,2$ in the domain $X_{ij} \in [-1,1]$ 
and group them into **two bins**, labeling them
$y_i[0,1]$.

The $X_{ij}$ are our **features**, the $y_i$ our **labels**.

In [ ]:

# create sample data
np.random.seed(112)
n=60
# features
X=np.zeros(2*n).reshape(n,2)
X[:,0] = np.random.uniform(-1,1,n)
X[:,1] = np.random.uniform(-1,1,n)
# labels
y=np.ones(n,dtype=int)
for i in range(n):
    if (X[i,0] <= 0.4):
        y[i] = 0
#print(y)
col_list = [cmap_set1(y[i]) for i in range(n)]

fig,ax = plt.subplots(1, 1,figsize=(6,6))
plt.xlim([-1.1,1.1]);plt.xlabel('x$_1$')
plt.ylim([-1.1,1.1]);plt.ylabel('x$_2$')
ax.scatter(X[:,0],X[:,1],c=col_list,s=100)
plt.show()

----
## Method: Logistic regression

Define **new** point $x^0$ and figure out, to which group of points it belongs,
using a **logistic regression**.

The red dots have the label $y=0$, the blue dots the label $y=1$.
There is a sharp gradient between the two clusters.

We guess a **straight line**, $y=a+b x$, which best separates the two labels. We use
$$
\begin{array}{rcl}
a &=& -7 \\ b&=&20
\end{array}
$$
The black line above has been drawn with these two parameter values.

In [ ]:
x0 = np.array([0.45,0.52])
a = -7.00
b = 20.
def straight_line(a, b, x):
    y = a + b*x
    return y
xx=np.linspace(-1,1,41)
print('intercept: ',round(a,2))
print('slope:     ',round(b,2))


fig,ax = plt.subplots(1, 1,figsize=(8,8))
plt.xlim([-1.1,1.1]);plt.xlabel('x$_1$')
plt.ylim([-1.1,1.1]);plt.ylabel('x$_2$')
ax.scatter(X[:,0],X[:,1],c=col_list,s=100)
ax.scatter(x0[0],x0[1],c='g',s=300,edgecolor='black')
plt.plot(xx,straight_line(a,b,xx),linewidth=2,linestyle='-',color='black')
plt.show()

We use a **sigmoid function** to separate the two clusters with discrete values (classification problem),
because it mimicks the bindary behaviour of the labels:
$$
S(p) = \frac{1}{1+e^{-p}}
$$
For $p < 0$, the label is $y=0$, for $p > 0$, the label is $y=1$.

In [ ]:
def sigmoid(p):
    sigmoid = 1. / (1.+np.exp(-p))
    return sigmoid

p = np.linspace(-10,10,21)
plt.plot(p,sigmoid(p))

With our two features, we need to fit, as a linear approximation, a **plane** through the
features $(x_1,x_2)$:
$$
p_i(a_j) = a_0 + a_1 x_1 + a_2 x_2
$$
Using the **coefficient vector** $a_j$, we rewrite:
$$
p_i(a_j) = \left( 1,x_1,x_2 \right) 
\left( \begin{array}{c} a_0 \\ a_1 \\ a_2 \end{array} \right)
$$
We thus have to **extend** our feature matrix with a constant term ("1")...

In our example of the fitted straight line above, we identify:
$$
\begin{array}{rcl}
a_0 &=& a \\
a_1 &=& b \\
a_2 &=& -1
\end{array}
$$
We use the **sigmoid function** with $\eta$ defined above for our sample data set:

In [ ]:
coeff = [a,b,-1]
xx=np.linspace(-1,1,41)
XX,YY=np.meshgrid(xx,xx)
p=coeff[0] + coeff[1]*XX + coeff[2]*YY
ZZ=sigmoid(p)
print(ZZ.min())
print(ZZ.max())

fig,ax = plt.subplots(1, 1,figsize=(9,8))
plt.xlim([-1,1]);plt.xlabel('x$_1$')
plt.ylim([-1,1]);plt.ylabel('x$_2$')
CS=plt.contourf(XX,YY,ZZ,alpha=0.3)
plt.colorbar(CS,ticks=[0.,0.5,1.0])
ax.scatter(X[:,0],X[:,1],c=col_list,s=100)
ax.scatter(x0[0],x0[1],c='r',s=300,edgecolor='black')
plt.plot(xx,straight_line(a,b,xx),linewidth=2,linestyle='-',color='black')
plt.show()

But how do we find the coefficients $a_i$?
1. Define the **extended** feature matrix:
$$
X_{ij}^{poly} = 
\left(
\begin{array}{ccc}
1 & x_1^1 & x_2^1 \\
1 & x_1^2 & x_2^2 \\
  & \vdots &\\
1 & x_1^n & x_2^n \\
\end{array}
\right)
$$
2. Define the **coefficient vector**: $a_j = (a_0,a_1,a_2)$
3. Loop, until convergence achieved:
    - create a **prediction function**: $ p_i = X_{ij}^{poly} \cdot a_j$.
    - Calculate the **probability**: $P=S(p)$.
    - Calculate the **cost** function:
    $$
    \chi = \frac{1}{m} \sum_n \left[ y_i \log(P_i) + (1-y_i) \ln(1-P_i) \right]
    $$
    - Calculate the **gradient** of the cost function:
    $$
    \frac{\partial \chi}{\partial a_j} =
    \frac{1}{m} X_{ij} \cdot S(p_i - y_i)
    $$
    - Improve the **coefficient vector**:
    $$
    a_j = a_j - \alpha \frac{\partial \chi}{\partial a_j}
    $$
    with $\alpha$ a learning rate.

In [ ]:
def prediction(theta, x):
    # Computes the weighted sum of inputs
    return np.dot(x, theta)

def probability(theta, x):
    # Returns the probability after passing through sigmoid
    return sigmoid(prediction(theta, x))

def cost_function(theta, x, y):
    # Computes the cost function for all the training samples
    m = x.shape[0]
    total_cost = -(1 / m) * np.sum(
        y * np.log(probability(theta, x)) + (1 - y) * np.log(
            1 - probability(theta, x)))
    return total_cost

def gradient(theta, x, y):
    # Computes the gradient of the cost function at the point theta
    m = x.shape[0]
    return (1 / m) * np.dot(x.T, sigmoid(prediction(theta,   x)) - y)

In [ ]:
Xpoly = np.c_[np.ones((X.shape[0], 1)), X]
theta = np.zeros((Xpoly.shape[1], 1))

In [ ]:
opt_weights = scipy.optimize.fmin_tnc(func=cost_function, x0=theta,
                  fprime=gradient,args=(Xpoly, y))
parameters = opt_weights[0]
print(parameters)
print('intercept: ',round(parameters[0],2))
print('slope:     ',round(parameters[1],2))

In [ ]:
print(y.shape)

In [ ]:
x_values = [np.min(X[:, 0] - 0.1), np.max(X[:, 1] + 0.1)]
y_values = - (parameters[0] + np.dot(parameters[1], x_values)) / parameters[2]

fig,ax = plt.subplots(1, 1,figsize=(9,8))
plt.xlim([-1,1]);plt.xlabel('x$_1$')
plt.ylim([-1,1]);plt.ylabel('x$_2$')
ax.scatter(X[:,0],X[:,1],c=col_list,s=100)
ax.scatter(x0[0],x0[1],c='g',s=300,edgecolor='black')

plt.plot(x_values, y_values,linewidth=2,color='black',label='Decision Boundary')
plt.show()

Predictor for the logistic regression:

In [ ]:
x0poly = np.r_[1.,x0].reshape(1,3)
x1 = np.array([0.25,0.25])
x1poly  = np.r_[1.,x1].reshape(1,3)

#print(Xpoly)

def logr_predict(parameters, Xpoly):
    pp = probability(parameters, Xpoly)
    labels = np.zeros(len(pp),dtype='int')
    for i in range(len(pp)):
        if (pp[i]>0.50):
            labels[i] = 1
    return labels
    
labelx0=logr_predict(parameters, x0poly)
labelx1=logr_predict(parameters, x1poly)
print('x0:    ',x0poly)
print('label: ',labelx0)
print('label: ',labelx1)

In [ ]:
x_values = [np.min(X[:, 0] - 0.1), np.max(X[:, 1] + 0.1)]
y_values = - (parameters[0] + np.dot(parameters[1], x_values)) / parameters[2]

fig,ax = plt.subplots(1, 1,figsize=(9,8))
plt.xlim([-1,1]);plt.xlabel('x$_1$')
plt.ylim([-1,1]);plt.ylabel('x$_2$')
ax.scatter(X[:,0],X[:,1],c=col_list,s=100)
ax.scatter(x0[0],x0[1],c=cmap_set1.colors[labelx0],s=300,edgecolor='black')
ax.scatter(x1[0],x1[1],c=cmap_set1.colors[labelx1],s=300,edgecolor='black')

plt.plot(x_values, y_values,linewidth=2,color='black',label='Decision Boundary')
plt.show()

... done